In [2]:
import pandas as pd
import numpy as np

# Load raw loan data
df = pd.read_csv("raw_loan_data.csv")

# --------------------------------------------------
# Create Risk Tier
# --------------------------------------------------

def assign_risk_tier(fico_score):
    if fico_score >= 720:
        return "Low Risk"
    elif fico_score >= 660:
        return "Medium Risk"
    else:
        return "High Risk"

df["risk_tier"] = df["fico_score"].apply(assign_risk_tier)

# --------------------------------------------------
# Create Loan Status (Default / Fully Paid)
# --------------------------------------------------

default_probability = (
    0.05
    + ((700 - df["fico_score"]).clip(lower=0) * 0.0004)
    + (df["dti"] * 0.15)
)

# Small Business loans are slightly riskier
default_probability += np.where(
    df["purpose"] == "Small Business",
    0.05,
    0
)

# Higher interest rates indicate higher risk
default_probability += np.where(
    df["int_rate"] > 18,
    0.03,
    0
)

# Keep probability within realistic limits
default_probability = default_probability.clip(0.05, 0.60)

# Generate final loan status
df["loan_status"] = np.where(
    np.random.rand(len(df)) < default_probability,
    "Default",
    "Fully Paid"
)

# --------------------------------------------------
# Save Final Dataset
# --------------------------------------------------

df.to_csv("loan_data_clean.csv", index=False)

# --------------------------------------------------
# Quick Validation
# --------------------------------------------------

print("\nLoan Status Distribution")
print(df["loan_status"].value_counts())

print("\nRisk Tier Distribution")
print(df["risk_tier"].value_counts())

print("\nDataset Shape")
print(df.shape)

print("\nFeature Engineering Completed Successfully!")


Loan Status Distribution
loan_status
Fully Paid    22155
Default        2845
Name: count, dtype: int64

Risk Tier Distribution
risk_tier
Medium Risk    11007
High Risk       8539
Low Risk        5454
Name: count, dtype: int64

Dataset Shape
(25000, 12)

Feature Engineering Completed Successfully!


In [3]:
import os
print(os.listdir())

['loan_data_clean.csv', '.ipynb_checkpoints', 'feature_engineering.py.ipynb', 'Credit_Risk_Data_Gen.ipynb', 'raw_loan_data.csv']


In [4]:
df = pd.read_csv("loan_data_clean.csv")
df.head()

,loan_id,customer_id,funded_amount,term_months,int_rate,fico_score,dti,purpose,home_ownership,issue_date,risk_tier,loan_status
0,LN100000,CUST210477,10000,36,18.20,679,0.29,Credit Card Refinance,RENT,2024-07-24,Medium Risk,Fully Paid
1,LN100001,CUST201825,50000,36,15.07,740,0.48,Home Improvement,OWN,2024-01-16,Low Risk,Fully Paid
2,LN100002,CUST200410,20000,36,16.50,711,0.11,Debt Consolidation,RENT,2024-04-14,Medium Risk,Fully Paid
3,LN100003,CUST212150,15000,60,17.23,639,0.32,Credit Card Refinance,OWN,2025-04-18,High Risk,Fully Paid
4,LN100004,CUST204507,5000,36,13.09,769,0.19,Debt Consolidation,MORTGAGE,2024-09-30,Low Risk,Fully Paid
